# LDA Preprocessing

In this section, I load my data and perform basic exploratory analysis & preprocess my text as best I can for future 

In [1]:
# import necessary libraries
%pip install pandas numpy nltk scikit-learn
import pandas as pd
import numpy as np

Note: you may need to restart the kernel to use updated packages.


In [2]:
# read in datafile
twitter_data_df = pd.read_csv('Mental-Health-Twitter.csv')

# ensure data loaded properly
twitter_data_df.head()

,Unnamed: 0,post_id,post_created,post_text,user_id,followers,friends,favourites,statuses,retweets,label
0,0,637894677824413696,Sun Aug 30 07:48:37 +0000 2015,It's just over 2 years since I was diagnosed w...,1013187241,84,211,251,837,0,1
1,1,637890384576778240,Sun Aug 30 07:31:33 +0000 2015,"It's Sunday, I need a break, so I'm planning t...",1013187241,84,211,251,837,1,1
2,2,637749345908051968,Sat Aug 29 22:11:07 +0000 2015,Awake but tired. I need to sleep but my brain ...,1013187241,84,211,251,837,0,1
3,3,637696421077123073,Sat Aug 29 18:40:49 +0000 2015,RT @SewHQ: #Retro bears make perfect gifts and...,1013187241,84,211,251,837,2,1
4,4,637696327485366272,Sat Aug 29 18:40:26 +0000 2015,It’s hard to say whether packing lists are mak...,1013187241,84,211,251,837,1,1


## Basic EDA

_______

In [3]:
# sanity check the data
# shape of the datafram
twitter_data_df.shape

(20000, 11)

In [4]:
# check head again
twitter_data_df.head()

,Unnamed: 0,post_id,post_created,post_text,user_id,followers,friends,favourites,statuses,retweets,label
0,0,637894677824413696,Sun Aug 30 07:48:37 +0000 2015,It's just over 2 years since I was diagnosed w...,1013187241,84,211,251,837,0,1
1,1,637890384576778240,Sun Aug 30 07:31:33 +0000 2015,"It's Sunday, I need a break, so I'm planning t...",1013187241,84,211,251,837,1,1
2,2,637749345908051968,Sat Aug 29 22:11:07 +0000 2015,Awake but tired. I need to sleep but my brain ...,1013187241,84,211,251,837,0,1
3,3,637696421077123073,Sat Aug 29 18:40:49 +0000 2015,RT @SewHQ: #Retro bears make perfect gifts and...,1013187241,84,211,251,837,2,1
4,4,637696327485366272,Sat Aug 29 18:40:26 +0000 2015,It’s hard to say whether packing lists are mak...,1013187241,84,211,251,837,1,1


In [5]:
# check tail
twitter_data_df.tail()

,Unnamed: 0,post_id,post_created,post_text,user_id,followers,friends,favourites,statuses,retweets,label
19995,19995,819336825231773698,Thu Jan 12 00:14:56 +0000 2017,A day without sunshine is like night.,1169875706,442,230,7,1063601,0,0
19996,19996,819334654260080640,Thu Jan 12 00:06:18 +0000 2017,"Boren's Laws: (1) When in charge, ponder. (2) ...",1169875706,442,230,7,1063601,0,0
19997,19997,819334503042871297,Thu Jan 12 00:05:42 +0000 2017,The flow chart is a most thoroughly oversold p...,1169875706,442,230,7,1063601,0,0
19998,19998,819334419374899200,Thu Jan 12 00:05:22 +0000 2017,"Ships are safe in harbor, but they were never ...",1169875706,442,230,7,1063601,0,0
19999,19999,819334270825197568,Thu Jan 12 00:04:47 +0000 2017,Black holes are where God is dividing by zero.,1169875706,442,230,7,1063601,0,0


In [6]:
# get column names and types
twitter_data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Unnamed: 0    20000 non-null  int64
 1   post_id       20000 non-null  int64
 2   post_created  20000 non-null  str  
 3   post_text     20000 non-null  str  
 4   user_id       20000 non-null  int64
 5   followers     20000 non-null  int64
 6   friends       20000 non-null  int64
 7   favourites    20000 non-null  int64
 8   statuses      20000 non-null  int64
 9   retweets      20000 non-null  int64
 10  label         20000 non-null  int64
dtypes: int64(9), str(2)
memory usage: 1.7 MB


In [7]:
# get descriptive statistics
twitter_data_df.describe()

,Unnamed: 0,post_id,user_id,followers,friends,favourites,statuses,retweets,label
count,20000.000000,2.000000e+04,2.000000e+04,20000.000000,20000.000000,20000.000000,2.000000e+04,20000.000000,20000.000000
mean,9999.500000,6.874728e+17,3.548623e+16,900.483950,782.428750,6398.235550,4.439442e+04,1437.927300,0.500000
std,5773.647028,1.708396e+17,1.606083e+17,1899.913961,1834.817945,8393.072914,1.407785e+05,15119.665118,0.500013
min,0.000000,3.555966e+09,1.472438e+07,0.000000,0.000000,0.000000,3.000000e+00,0.000000,0.000000
25%,4999.750000,5.931686e+17,3.242944e+08,177.000000,211.000000,243.000000,5.129000e+03,0.000000,0.000000
50%,9999.500000,7.637400e+17,1.052122e+09,476.000000,561.000000,2752.000000,1.325100e+04,0.000000,0.500000
75%,14999.250000,8.153124e+17,2.285923e+09,1197.000000,701.000000,8229.000000,5.289200e+04,1.000000,1.000000
max,19999.000000,8.194574e+17,7.631825e+17,28614.000000,28514.000000,39008.000000,1.063601e+06,839540.000000,1.000000


In [8]:
# check to see potential missing values
twitter_data_df.isnull().sum()

Unnamed: 0      0
post_id         0
post_created    0
post_text       0
user_id         0
followers       0
friends         0
favourites      0
statuses        0
retweets        0
label           0
dtype: int64

--------

## Preprocessing

Currently using NLTK to remove stopwords. but may move over to SpaCy if improvement is needed.

resource: https://www.geeksforgeeks.org/nlp/removing-stop-words-nltk-python/

resource: https://www.geeksforgeeks.org/nlp/topic-modeling-using-latent-dirichlet-allocation-lda/

In [9]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize 

# download stopwords
nltk.download('stopwords')
nltk.download('punkt')
stop_words = stopwords.words('english')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\leslie\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\leslie\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [10]:
# clean the text data
import re

def clean_and_lower(text):
    # remove urls
    text = re.sub(r'http\S+|www\S+', '', text)
    # remove mentions
    text = re.sub(r'@\w+', '', text)
    # remove RT (retweets)
    text = re.sub(r'\bRT\b', '', text)
    # whitespace removal
    text = re.sub('\s+', ' ', text)
    # email removal
    text = re.sub('\S*@\S*\s?', '', text)
    # apostrophe removal
    text = re.sub('\'', '', text)
    # remove non-alphanumeric characters
    text = re.sub('[^a-zA-Z0-9\s]', '', text)
    return text.lower().strip()

twitter_data_df['cleaned_text'] = twitter_data_df['post_text'].apply(clean_and_lower)

<>:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:14: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:18: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:14: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:18: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\leslie\AppData\Local\Temp\ipykernel_29616\2381581573.py:12: SyntaxW

In [11]:
twitter_data_df['cleaned_text'].head()

0    its just over 2 years since i was diagnosed wi...
1    its sunday i need a break so im planning to sp...
2    awake but tired i need to sleep but my brain h...
3    retro bears make perfect gifts and are great f...
4    its hard to say whether packing lists are maki...
Name: cleaned_text, dtype: str

In [12]:
import nltk
nltk.download('punkt_tab')

def tokenize_and_remove_stopwords(text):
    tokenized_tweet = word_tokenize(text)
    nostopword_tweets = [word for word in tokenized_tweet if word not in stop_words]
    return " ".join(nostopword_tweets)

tokenized_cleaned_tweets = [tokenize_and_remove_stopwords(tweet) for tweet in twitter_data_df['cleaned_text']]

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\leslie\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [13]:
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

lemmatized_tweets = [lemmatize_text(tweet) for tweet in tokenized_cleaned_tweets]

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\leslie\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Vectorizing and Tokens

## Create a token filter to only keep meaningful tokens and improve performance

In [14]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [15]:
vectorizer = CountVectorizer(
    min_df=5,  # ignore tokens in less than 5 tweets,
    max_df=0.5,  # ignore tokens in more than 50% of tweets
    max_features=10000,  # limit to top 100000 tokens
    token_pattern=r'\b[a-zA-Z]{3,}\b' # only tokens longer than 3 characters
)

dtm = vectorizer.fit_transform(lemmatized_tweets)
print(f'Tweet Matrix shape: {dtm.shape}')
print(f'Vocab Size: {len(vectorizer.vocabulary_)}')

Tweet Matrix shape: (20000, 3531)
Vocab Size: 3531


# Potential Model

Currently, this is a super base level model based on online example implementations of LDA using sklearn. Will continue developing on this model and it's parameters to improve performance.

In [16]:
lda = LatentDirichletAllocation(
    n_components=10, # topic count for current testing
    random_state=0)

lda.fit(dtm)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",10
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [17]:
# see what the top words are for each topic observed in initial test
feature_names = vectorizer.get_feature_names_out()
n_top_words = 10
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
    print(f'Topic {topic_idx}: {" | ".join(top_words)}')

Topic 0: best | god | please | video | zayn | bestmusicvideo | pillowtalk | iheartawards | today | wow
Topic 1: hey | thanks | follow | mental | would | people | yes | ive | want | know
Topic 2: dont | know | like | one | people | see | lol | time | girl | mean
Topic 3: yong | wearepayting | foryong | money | paytforluckysun | thats | going | damn | like | really
Topic 4: depression | say | treatment | thank | twitter | following | hello | overcome | migraine | anytime
Topic 5: feel | fuck | get | good | never | dont | like | let | hate | family
Topic 6: day | love | friend | great | cant | positive | shit | thinking | stop | one
Topic 7: life | back | need | game | get | hour | right | christmas | better | make
Topic 8: man | via | still | trump | thing | life | put | get | amp | think
Topic 9: like | look | new | happy | year | talk | one | day | good | ill


In [22]:
%pip install matplotlib wordcloud
import matplotlib.pyplot as plt
from wordcloud import WordCloud

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


  Using cached matplotlib-3.10.8-cp314-cp314-win_amd64.whl.metadata (52 kB)
  Using cached wordcloud-1.9.6-cp314-cp314-win_amd64.whl.metadata (3.5 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp314-cp314-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp314-cp314-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.8-cp314-cp314-win_amd64.whl (8.3 MB)
Using cached wordcloud-1.9.6-cp314-cp314-win_amd64.whl (308 kB)
Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl (232 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.62.1-cp314-cp314-win_amd64.whl (2.3 MB)
Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl (75 kB)
Using cached pillow-12.2.0-cp314-cp314-win_

In [ ]:
# Visualize topics using word clouds
for t in range(lda.n_components):
    plt.figure(figsize=(10, 6))
    # Get top words for each topic
    top_words_idx = lda.components_[t].argsort()[-50:]
    top_words = {feature_names[i]: lda.components_[t][i] for i in top_words_idx}
    # Generate word cloud from topic frequencies
    plt.imshow(WordCloud(background_color="white").fit_words(top_words))
    plt.axis("off")
    plt.title("Topic #" + str(t))
    plt.show()


AttributeError: 'LatentDirichletAllocation' object has no attribute 'num_topics'